In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import time
from collections import defaultdict

prereq = "para trabajar pre req progv22025-09-26160704_procesado_v3.xlsx"
#historial = "detalle matricula cohorte 2019.xlsx"
#historial = "detalle matricula cohorte 2019-2025.xlsx"
historial = "detalle matricula cohorte 2014-2025.csv"
                                

In [2]:
def evaluar_prerrequisitos_mejorado(df_historial: pd.DataFrame, df_prereq: pd.DataFrame, debug_max_filas: int = 5) -> pd.DataFrame:
    """
    Evalúa prerrequisitos para cada fila del historial académico y devuelve el historial
    con columnas Prereq_i_* + Observacion_Prerrequisito.

    Reglas clave:
      - Aprobación = Materia_Aprobada == 'Y' (sin importar nota).
      - Temporal: aprobación debe ser en Periodo < Periodo de la materia.
      - Selección de opción:
          1) Prioriza opciones COMPLETAMENTE cumplidas.
          2) Entre opciones completas: prioriza las que tienen múltiples códigos (&) y luego menor número de opción.
          3) Si no hay completas, elige la mejor PARCIAL (más códigos aprobados; mismo desempate).
      - Si no hay opciones → "No tiene pre requisito".
      - Si hay opciones pero ninguna cumple ni parcialmente → "Tiene pre requisito pero no cumple".
      - No repetir códigos en la opción elegida (dedup ordenado).
      - Nivel=1, Tipo='Directo', EsCadena='FALSO' (no se construyen cadenas en esta función).

    Parámetros:
      - debug_max_filas: cuántas filas del historial mostrar con prints detallados (para depuración).
    """

    print("=== INICIO evaluar_prerrequisitos_mejorado ===")
    print(f"Historial: {df_historial.shape[0]} filas, {df_historial.shape[1]} cols")
    print(f"Prerrequisitos: {df_prereq.shape[0]} filas, {df_prereq.shape[1]} cols")

    # -------- Normalización mínima --------
    H = df_historial.copy()
    H['Periodo'] = pd.to_numeric(H['Periodo'], errors='coerce').astype('Int64')
    H['Materia_Aprobada'] = H['Materia_Aprobada'].astype(str).str.upper().fillna('')

    # Índice rápido por (Pidm, Cod, Periodo) ordenado
    H_sorted = H.sort_values(['Pidm','Cod materia curso','Periodo']).reset_index(drop=True)
    grp = H_sorted.groupby(['Pidm','Cod materia curso'], sort=False)

    def obtener_estado(pidm, cod, periodo_lim):
        """Devuelve (nota_ultima, intentos, aprobado_bool) considerando Periodo < periodo_lim."""
        try:
            g = grp.get_group((pidm, cod))
        except KeyError:
            return (None, 0, False)
        g_pre = g[g['Periodo'] < periodo_lim]
        if g_pre.empty:
            return (None, 0, False)
        ultimo = g_pre.iloc[-1]
        nota = ultimo.get('Calificacion_Final', None)
        intentos = len(g_pre)
        aprobado = (ultimo['Materia_Aprobada'] == 'Y')
        return (nota, intentos, aprobado)

    # -------- Construcción de opciones desde df_prereq --------
    patron = r'^Opcion_Prereq_\d+$'
    cols_opc = [c for c in df_prereq.columns if re.match(patron, c)]
    cols_opc.sort(key=lambda x: int(x.split('_')[-1]))
    print(f"Opciones detectadas: {cols_opc}")

    if not cols_opc:
        print("No hay columnas Opcion_Prereq_* en df_prereq → marcar todo como 'No tiene pre requisito'")
        out = H.copy()
        out['Observacion_Prerrequisito'] = 'No tiene pre requisito'
        return out

    PR = df_prereq.copy()
    PR_long = PR.melt(
        id_vars=['Smbarul_Key_Rule','Programa'],
        value_vars=cols_opc,
        var_name='Opcion',
        value_name='Prerrequisitos_Raw'
    ).dropna(subset=['Prerrequisitos_Raw'])

    if PR_long.empty:
        print("No hay valores en opciones de prerrequisito → marcar 'No tiene pre requisito'")
        out = H.copy()
        out['Observacion_Prerrequisito'] = 'No tiene pre requisito'
        return out

    # Parseo de códigos y metadata para orden estable
    def parse_lista(x):
        return [c.strip() for c in str(x).split('&') if str(c).strip()]

    PR_long['Codigos'] = PR_long['Prerrequisitos_Raw'].apply(parse_lista)
    PR_long['Opcion_Num'] = PR_long['Opcion'].str.extract(r'(\d+)').astype(int)
    PR_long['Es_Multi'] = PR_long['Codigos'].apply(lambda L: len(L) > 1)
    PR_long['RowOrder'] = np.arange(len(PR_long))  # orden estable para desempates

    print(f"Opciones en largo: {len(PR_long)} filas (materia/programa/opción)")

    # Index por materia para lookup rápido (trae TODAS las opciones de TODOS los programas)
    opciones_por_materia = defaultdict(list)
    for _, r in PR_long.iterrows():
        opciones_por_materia[r['Smbarul_Key_Rule']].append({
            'Programa'   : r['Programa'],
            'Opcion'     : r['Opcion'],
            'Opcion_Num' : int(r['Opcion_Num']),
            'Codigos'    : list(r['Codigos']),
            'Es_Multi'   : bool(r['Es_Multi']),
            'RowOrder'   : int(r['RowOrder'])
        })

    # -------- Evaluación fila a fila --------
    resultados = []
    n = len(H_sorted)
    print(f"Evaluando {n} filas del historial...")

    for idx, fila in H_sorted.iterrows():
        pidm    = fila['Pidm']
        materia = fila['Cod materia curso']
        periodo = int(fila['Periodo'])

        if idx < debug_max_filas:
            print(f"\n[DEBUG fila {idx}] Pidm={pidm}, Materia={materia}, Periodo={periodo}")

        opciones = opciones_por_materia.get(materia, [])
        if not opciones:
            # No hay prerrequisitos definidos para esta materia
            resultados.append((pidm, materia, periodo, 'No tiene pre requisito', []))
            if idx < debug_max_filas:
                print("  - Sin opciones en catálogo → No tiene pre requisito")
            continue

        # Orden de prioridad: MULTI primero, luego menor Opcion_Num, luego RowOrder (estable)
        opciones_ord = sorted(opciones, key=lambda d: (-int(d['Es_Multi']), int(d['Opcion_Num']), int(d['RowOrder'])))

        mejor_completa = None
        mejor_parcial  = None
        mejor_parcial_score = -1  # número de aprobados en esa opción

        # Recorremos opciones en orden de prioridad
        for op in opciones_ord:
            cods = op['Codigos']
            estado_codigos = []
            aprob_count = 0

            # Evaluación de cada código del prerrequisito
            for cod in cods:
                nota, intentos, aprobado = obtener_estado(pidm, cod, periodo)
                estado_codigos.append((cod, nota, intentos, aprobado))
                if aprobado:
                    aprob_count += 1

            # ¿Opción completamente cumplida?
            if aprob_count == len(cods):
                mejor_completa = (op, estado_codigos)
                if idx < debug_max_filas:
                    print(f"  - Opción COMPLETA encontrada: {op['Opcion']} ({'&' if op['Es_Multi'] else 'simple'}) → se elige y se detiene la búsqueda")
                break  # ya encontramos la mejor completa por nuestra prioridad

            # ¿Opción parcialmente cumplida? (al menos un aprobado)
            if aprob_count > 0 and aprob_count > mejor_parcial_score:
                mejor_parcial_score = aprob_count
                mejor_parcial = (op, estado_codigos)

        # Decidir resultado para la fila
        if mejor_completa is not None:
            op, lista = mejor_completa
            # Deduplicar códigos preservando orden
            vistos = set()
            lista_unica = []
            for cod, nota, intentos, aprobado in lista:
                if cod not in vistos:
                    lista_unica.append((cod, nota, intentos, aprobado))
                    vistos.add(cod)
            resultados.append((pidm, materia, periodo, 'Prerrequisito cumplido', lista_unica))
        elif mejor_parcial is not None:
            op, lista = mejor_parcial
            vistos = set()
            lista_unica = []
            for cod, nota, intentos, aprobado in lista:
                if cod not in vistos:
                    lista_unica.append((cod, nota, intentos, aprobado))
                    vistos.add(cod)
            resultados.append((pidm, materia, periodo, 'Cumple parcialmente el requisito', lista_unica))
            if idx < debug_max_filas:
                print(f"  - Mejor opción PARCIAL: {op['Opcion']} con {mejor_parcial_score} aprobados")
        else:
            resultados.append((pidm, materia, periodo, 'Tiene pre requisito pero no cumple', []))
            if idx < debug_max_filas:
                print("  - Ninguna opción cumple ni parcialmente → Tiene pre requisito pero no cumple")

        # Progreso
        if (idx + 1) % 10000 == 0:
            print(f"  ...procesadas {idx+1} filas")

    # -------- Construcción de salida en ancho --------
    filas_out = []
    for pidm, materia, periodo, obs, prereqs in resultados:
        fila = {
            'Pidm': pidm,
            'Cod materia curso': materia,
            'Periodo': periodo,
            'Observacion_Prerrequisito': obs
        }
        # Solo los de la opción elegida (completa o parcial). Sin duplicados.
        for i, (codigo, nota, intentos, aprobado) in enumerate(prereqs, start=1):
            fila[f'Prereq_{i}_Codigo']    = codigo
            fila[f'Prereq_{i}_Nota']      = nota
            fila[f'Prereq_{i}_Intentos']  = intentos
            fila[f'Prereq_{i}_Nivel']     = 1
            fila[f'Prereq_{i}_Tipo']      = 'Directo'
            fila[f'Prereq_{i}_EsCadena']  = 'FALSO'
        filas_out.append(fila)

    df_result = pd.DataFrame(filas_out)
    print(f"\nFilas con evaluación: {df_result.shape[0]}. Columnas generadas (máx): {df_result.shape[1]}")

    # Métricas de observación
    if 'Observacion_Prerrequisito' in df_result.columns:
        print("\nResumen Observacion_Prerrequisito:")
        print(df_result['Observacion_Prerrequisito'].value_counts(dropna=False))

    # -------- Merge no destructivo: historial + nuevas columnas --------
    df_final = H.merge(df_result, on=['Pidm','Cod materia curso','Periodo'], how='left')
    print("=== FIN evaluar_prerrequisitos_mejorado ===\n")
    return df_final



In [3]:
def guardar_resultados(df):
        """Guarda los resultados en un archivo Excel"""
        if df is None:
            print("No hay resultados para guardar")
            return

        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        #nombre_archivo = "Resultados v6/resultados_prerrequisitos_v6_cadenas_"+timestamp+".xlsx"
        nombre_archivo = "Resultados v6/resultados_prerrequisitos_v6_cadenas_"+timestamp+".csv"

        try:
            # Limpiar columnas completamente vacías
            df = df.dropna(axis=1, how='all')
            print(f"✓ Guardando resultados en: {nombre_archivo} ...")
            #df.to_excel(nombre_archivo, index=False)
            df.to_csv(nombre_archivo, index=False)
            print(f"✓ Resultados guardados en: {nombre_archivo}")
            print(f"Columnas en el archivo: {len(df.columns)}")
        except Exception as e:
            print(f"Error al guardar resultados: {e}")

In [4]:
def evaluar_prerrequisitos_con_cadena(df_historial: pd.DataFrame,
                                      df_prereq: pd.DataFrame,
                                      debug_max_filas: int = 5,
                                      debug_imprime_n: int = 10) -> pd.DataFrame:
    """
    Evalúa prerrequisitos directos y en cadena (hasta 'debug_max_filas' niveles)
    para cada fila del historial. Devuelve el historial expandido con:

      - Observacion_Prerrequisito ∈ {
          'Prerrequisito cumplido',
          'Cumple parcialmente el requisito',
          'Tiene pre requisito pero no cumple',
          'No tiene pre requisito'
        }
      - Prereq_i_Codigo / Nota / Intentos / Nivel / Tipo / EsCadena

    Reglas:
      - Aprobación: Materia_Aprobada == 'Y' (nota no define aprobación).
      - Temporal: la aprobación válida debe estar en Periodo < Periodo de la materia raíz.
      - Selección de opción:
          1) Primero opciones COMPLETAS; prioriza las que tienen '&'.
          2) Si no hay completas → mejor PARCIAL (más aprobados; empate: '&' > simple > menor Opcion_Num).
          3) Si no hay completas ni parciales → “Tiene pre requisito pero no cumple”.
      - Cadena: se expande recursivamente por la opción ELEGIDA de cada curso (directo/cadena),
        sin repetir códigos (dedup por materia raíz).
    """

    print("=== INICIO evaluar_prerrequisitos_con_cadena ===")
    print(f"Historial: {df_historial.shape[0]} filas, {df_historial.shape[1]} cols")
    print(f"Prerrequisitos: {df_prereq.shape[0]} filas, {df_prereq.shape[1]} cols")

    # ---------------- Normalización mínima ----------------
    H = df_historial.copy()
    H['Periodo'] = pd.to_numeric(H['Periodo'], errors='coerce').astype('Int64')
    H['Materia_Aprobada'] = H['Materia_Aprobada'].astype(str).str.upper().fillna('')

    # Índice rápido por (Pidm, Cod, Periodo) ordenado
    H_sorted = H.sort_values(['Pidm','Cod materia curso','Periodo']).reset_index(drop=True)
    grp = H_sorted.groupby(['Pidm','Cod materia curso'], sort=False)

    def obtener_estado(pidm, cod, periodo_lim):
        """Devuelve (nota_ultima, intentos, aprobado_bool) considerando Periodo < periodo_lim."""
        try:
            g = grp.get_group((pidm, cod))
        except KeyError:
            return (None, 0, False)
        g_pre = g[g['Periodo'] < periodo_lim]
        if g_pre.empty:
            return (None, 0, False)
        ultimo = g_pre.iloc[-1]
        nota = ultimo.get('Calificacion_Final', None)
        intentos = len(g_pre)
        aprobado = (ultimo['Materia_Aprobada'] == 'Y')
        return (nota, intentos, aprobado)

    # ---------------- Catálogo de opciones (melt + parse) ----------------
    patron = r'^Opcion_Prereq_\d+$'
    cols_opc = [c for c in df_prereq.columns if re.match(patron, c)]
    cols_opc.sort(key=lambda x: int(x.split('_')[-1]))
    print(f"Opciones detectadas: {cols_opc}")

    if not cols_opc:
        print("No hay columnas Opcion_Prereq_* en df_prereq → marcar todo como 'No tiene pre requisito'")
        out = H.copy()
        out['Observacion_Prerrequisito'] = 'No tiene pre requisito'
        return out

    PR = df_prereq.copy()
    PR_long = PR.melt(
        id_vars=['Smbarul_Key_Rule','Programa'],
        value_vars=cols_opc,
        var_name='Opcion',
        value_name='Prerrequisitos_Raw'
    ).dropna(subset=['Prerrequisitos_Raw'])

    if PR_long.empty:
        print("No hay valores en opciones de prerrequisito → marcar 'No tiene pre requisito'")
        out = H.copy()
        out['Observacion_Prerrequisito'] = 'No tiene pre requisito'
        return out

    # Parseo de códigos: "A & B" → ["A","B"]
    def parse_lista(x):
        return [c.strip() for c in str(x).split('&') if str(c).strip()]

    PR_long['Codigos']    = PR_long['Prerrequisitos_Raw'].apply(parse_lista)
    PR_long['Opcion_Num'] = PR_long['Opcion'].str.extract(r'(\d+)').astype(int)
    PR_long['Es_Multi']   = PR_long['Codigos'].apply(lambda L: len(L) > 1)
    PR_long['RowOrder']   = np.arange(len(PR_long))  # orden estable para desempates

    print(f"Opciones en largo (materia/programa/opción): {len(PR_long)} filas")

    # Index de opciones por materia (unimos todas las filas de todos los programas)
    opciones_por_materia = defaultdict(list)
    for _, r in PR_long.iterrows():
        opciones_por_materia[r['Smbarul_Key_Rule']].append({
            'Programa'   : r['Programa'],
            'Opcion'     : r['Opcion'],
            'Opcion_Num' : int(r['Opcion_Num']),
            'Codigos'    : list(r['Codigos']),
            'Es_Multi'   : bool(r['Es_Multi']),
            'RowOrder'   : int(r['RowOrder'])
        })

    # ---------------- Selección de opción (reglas de prioridad) ----------------
    def seleccionar_opcion(pidm, materia, periodo):
        """
        Devuelve (estado_global, lista_detalles) donde:
          - estado_global ∈ {"cumple","parcial","no","sin_opciones"}
          - lista_detalles = [(cod, nota, intentos, aprobado_bool), ...] de la opción ELEGIDA
            (o [] si "no" o "sin_opciones")
        """
        opciones = opciones_por_materia.get(materia, [])
        if not opciones:
            return "sin_opciones", []

        # Orden de prioridad: MULTI primero, luego menor Opcion_Num, luego RowOrder
        opciones_ord = sorted(opciones, key=lambda d: (-int(d['Es_Multi']), int(d['Opcion_Num']), int(d['RowOrder'])))

        mejor_completa = None
        mejor_parcial  = None
        mejor_parcial_score = -1

        for op in opciones_ord:
            cods = op['Codigos']
            detalles = []
            aprob_count = 0

            for cod in cods:
                nota, intentos, aprobado = obtener_estado(pidm, cod, periodo)
                detalles.append((cod, nota, intentos, aprobado))
                if aprobado:
                    aprob_count += 1

            if aprob_count == len(cods):
                # opción completamente cumplida → devolver de una
                return "cumple", detalles

            if aprob_count > 0 and aprob_count > mejor_parcial_score:
                mejor_parcial_score = aprob_count
                mejor_parcial = detalles

        if mejor_parcial is not None:
            return "parcial", mejor_parcial

        return "no", []

    # ---------------- Recursión para cadena ----------------
    def expandir_cadena(pidm, materia, periodo, nivel, visitados):
        """
        Expande recursivamente los prerrequisitos de 'materia' elegidos por las reglas,
        acumulando códigos (sin repetir) con metadata. Devuelve lista de tuplas:
          (codigo, nota, intentos, aprobado_bool, nivel, tipo, es_cadena)
        """
        if nivel > debug_max_filas:
            return []

        estado, detalles = seleccionar_opcion(pidm, materia, periodo)
        resultado_nivel = []

        if estado in ("cumple", "parcial"):
            # Añadir los códigos de la opción ELEGIDA de esta 'materia'
            for cod, nota, intentos, aprobado in detalles:
                if cod not in visitados:
                    resultado_nivel.append((cod, nota, intentos, aprobado,
                                            nivel,
                                            'Directo' if nivel == 1 else 'Cadena',
                                            'FALSO'   if nivel == 1 else 'VERDADERO'))
                    visitados.add(cod)

            # Expandir siguiente nivel para CADA código elegido
            for cod, _, _, _ in detalles:
                resultado_nivel.extend(expandir_cadena(pidm, cod, periodo, nivel+1, visitados))

        # Si no hay opción elegible (estado "no") o "sin_opciones", no hay cadena por esta rama
        return resultado_nivel

    # ---------------- Evaluación fila a fila ----------------
    resultados = []
    n = len(H_sorted)
    print(f"Evaluando {n} filas del historial...")

    for idx, row in H_sorted.iterrows():
        pidm    = row['Pidm']
        materia = row['Cod materia curso']
        periodo = int(row['Periodo'])

         # Progreso con porcentaje
        if (idx + 1) % 500 == 0 or (idx + 1) == n:  # ajusta 500 al intervalo que prefieras
            print(f"Revisado {idx+1} / {n} lineas ({(idx+1)/n:.1%})")

        if idx < debug_imprime_n:
            print(f"\n[DEBUG fila {idx}] Pidm={pidm}, Materia={materia}, Periodo={periodo}")

        # 1) Selección de opción al NIVEL 1 (para Observacion)
        estado_root, detalles_root = seleccionar_opcion(pidm, materia, periodo)
        if estado_root == "sin_opciones":
            obs = "No tiene pre requisito"
            lista_total = []
            if idx < debug_imprime_n:
                print("  - Sin opciones → No tiene pre requisito")
        elif estado_root == "cumple":
            obs = "Prerrequisito cumplido"
            if idx < debug_imprime_n:
                print(f"  - Opción nivel 1: COMPLETA (directos)")
            # 2) Expandir cadena desde la materia raíz (incluye directos y cadena)
            visitados = set()
            lista_total = expandir_cadena(pidm, materia, periodo, nivel=1, visitados=visitados)
        elif estado_root == "parcial":
            obs = "Cumple parcialmente el requisito"
            if idx < debug_imprime_n:
                print(f"  - Opción nivel 1: PARCIAL (directos)")
            visitados = set()
            lista_total = expandir_cadena(pidm, materia, periodo, nivel=1, visitados=visitados)
        else:  # "no"
            obs = "Tiene pre requisito pero no cumple"
            lista_total = []
            if idx < debug_imprime_n:
                print("  - Ninguna opción de nivel 1 cumple ni parcialmente")

        # 3) Quitar duplicados preservando orden (por seguridad)
        seen = set()
        lista_unica = []
        for item in lista_total:
            cod = item[0]
            if cod not in seen:
                lista_unica.append(item)
                seen.add(cod)

        # 4) Armar fila de salida
        out_row = row.to_dict()
        out_row['Observacion_Prerrequisito'] = obs

        for j, (codigo, nota, intentos, aprobado, nivel, tipo, es_cadena) in enumerate(lista_unica, start=1):
            out_row[f'Prereq_{j}_Codigo']    = codigo
            out_row[f'Prereq_{j}_Nota']      = nota
            out_row[f'Prereq_{j}_Intentos']  = intentos
            out_row[f'Prereq_{j}_Nivel']     = nivel
            out_row[f'Prereq_{j}_Tipo']      = tipo
            out_row[f'Prereq_{j}_EsCadena']  = es_cadena

        resultados.append(out_row)

        # Progreso
        if (idx + 1) % 10000 == 0:
            print(f"  ...procesadas {idx+1} filas")

    df_final = pd.DataFrame(resultados)
    print("\nResumen Observacion_Prerrequisito:")
    print(df_final['Observacion_Prerrequisito'].value_counts(dropna=False))
    print("=== FIN evaluar_prerrequisitos_con_cadena ===\n")
    return df_final



In [5]:
## Codigo para optimizar el comportamiento del proceso

import pandas as pd
import numpy as np

def compactar_dtypes(df_historial: pd.DataFrame, df_prereq: pd.DataFrame):
    # Activar Copy-on-Write (Pandas 2.x) → reduce copias
    pd.options.mode.copy_on_write = True

    H = df_historial.copy()
    P = df_prereq.copy()

    # Tipos numéricos compactos
    if 'Pidm' in H.columns:
        # usa Int32 si puede tener nulos, si no int32
        H['Pidm'] = pd.to_numeric(H['Pidm'], errors='coerce').astype('Int32')

    if 'Periodo' in H.columns:
        H['Periodo'] = pd.to_numeric(H['Periodo'], errors='coerce').astype('Int32')

    if 'Calificacion_Final' in H.columns:
        H['Calificacion_Final'] = pd.to_numeric(H['Calificacion_Final'], errors='coerce').astype('float32')

    # Cadenas normalizadas y categorizadas (ahorra memoria)
    def _normcat(s):
        s = s.astype(str).str.strip()
        return s.astype('category')

    for col in ['Cod materia curso', 'Materia_Aprobada', 'Codigo_Programa']:
        if col in H.columns:
            if col == 'Materia_Aprobada':
                H[col] = H[col].astype(str).str.upper().str.strip().astype('category')
            else:
                H[col] = _normcat(H[col])

    for col in ['Smbarul_Key_Rule', 'Programa']:
        if col in P.columns:
            P[col] = _normcat(P[col])

    # Opciones (si existen) → strings limpios (no category para facilitar melt/parse)
    import re
    patron = r'^Opcion_Prereq_\d+$'
    cols_opc = [c for c in P.columns if re.match(patron, c)]
    for c in cols_opc:
        P[c] = P[c].astype(str).str.strip().replace({'nan': np.nan})

    return H, P


def particionar_por_pidm_balanceado(df_historial: pd.DataFrame, n_partes: int):
    # Contar filas por pidm para balancear la carga
    cnt = df_historial.groupby('Pidm', observed=True, sort=False).size().reset_index(name='rows')
    # Ordenar pidm de mayor a menor carga para greedy bin packing
    cnt = cnt.sort_values('rows', ascending=False).reset_index(drop=True)

    # Inicializar “buckets”
    buckets = [[] for _ in range(n_partes)]
    bucket_load = [0] * n_partes

    for _, r in cnt.iterrows():
        pidm = r['Pidm']
        rows = int(r['rows'])
        # colocar este pidm en el bucket más liviano
        k = int(np.argmin(bucket_load))
        buckets[k].append(pidm)
        bucket_load[k] += rows

    return buckets, bucket_load

# IMPORTANTE en Windows: pool necesita top-level functions
from functools import partial

# Variables globales para cada proceso (se asignan en el initializer)
_GLOBAL_DF_PREREQ = None

def _init_worker(df_prereq_global):
    # Guardar catálogo (solo lectura) en memoria de cada proceso
    global _GLOBAL_DF_PREREQ
    _GLOBAL_DF_PREREQ = df_prereq_global

def _procesar_particion(pidms_subset, df_historial_part, debug_max_filas, debug_imprime_n):
    """
    pidms_subset: lista de PIDM asignados a esta partición
    df_historial_part: DataFrame del historial completo (con __row_id),
                       se filtrará por esos PIDM.
    """
    # Filtrar historial por los pidm de esta partición
    H_part = df_historial_part[df_historial_part['Pidm'].isin(pidms_subset)].copy()

    # Llamar a tu función ya validada (no la tocamos)
    out = evaluar_prerrequisitos_con_cadena(
        df_historial=H_part,
        df_prereq=_GLOBAL_DF_PREREQ,
        debug_max_filas=debug_max_filas,
        debug_imprime_n=0  # 0 para no mezclar prints de miles de procesos
    )

    return out

import multiprocessing as mp
import math
import time

def ejecutar_en_paralelo(df_historial_raw: pd.DataFrame,
                         df_prereq_raw: pd.DataFrame,
                         n_procs: int = None,
                         debug_max_filas: int = 5):
    """
    Ejecuta evaluar_prerrequisitos_con_cadena en paralelo por particiones de PIDM.
    No cambia la lógica ni el resultado por fila, solo acelera.
    """
    if n_procs is None:
        # núcleos físicos aprox (si no conoces exacto, deja en cpu_count()//2 como base)
        n_procs = max(1, mp.cpu_count() // 2)

    print(f"\n=== Paralelizando por PIDM con {n_procs} procesos ===")

    # 1) Compactar dtypes (menos RAM, más rápido)
    H, P = compactar_dtypes(df_historial_raw, df_prereq_raw)

    # 2) Preservar orden original
    H = H.reset_index(drop=True).copy()
    H['__row_id'] = np.arange(len(H), dtype='int64')

    # 3) Particionar por Pidm
    start = time.time()
    buckets, loads = particionar_por_pidm_balanceado(H, n_partes=n_procs)
    print("Distribución de carga por proceso (filas):", loads, "→ total:", int(sum(loads)))
    print(f"Creación de particiones: {time.time() - start:.1f}s")

    # 4) Multiprocessing
    start = time.time()
    with mp.get_context("spawn").Pool(
        processes=n_procs,
        initializer=_init_worker,
        initargs=(P,),  # broadcast del catálogo a cada proceso (solo lectura)
        maxtasksperchild=50  # reciclar procesos para evitar leaks en jobs muy largos
    ) as pool:
        # Parciales: cada proceso recibe su lista de PIDMs y el mismo historial (filtrará internamente)
        tasks = []
        for k in range(n_procs):
            pidms_k = buckets[k]
            if not pidms_k:
                continue
            tasks.append(
                pool.apply_async(
                    _procesar_particion,
                    args=(pidms_k, H, debug_max_filas, 0)
                )
            )

        # Recolectar resultados con progreso simple
        parts = []
        for i, t in enumerate(tasks, 1):
            part = t.get()
            parts.append(part)
            elapsed = time.time() - start
            print(f"  → Partición {i}/{len(tasks)} lista ({len(part)} filas) - {elapsed:.1f}s")

    # 5) Unir, reordenar y devolver
    out = pd.concat(parts, ignore_index=True)

    # Reordenar al orden original
    if '__row_id' in out.columns:
        out = out.sort_values('__row_id').drop(columns=['__row_id']).reset_index(drop=True)

    total_elapsed = time.time() - start
    print(f"Tiempo total en paralelo (sin contar pre-partición): {total_elapsed:.1f}s")
    return out



In [6]:
print("=== ANALIZADOR OPTIMIZADO DE PRERREQUISITOS CON CADENAS ===\n")

# Solicitar archivo de prerrequisitos
while True:
    try:
        ruta_prereq = prereq #input("Ingrese la ruta del archivo 'pre_requisitos.xlsx': ").strip()
        df_prereq = pd.read_excel(ruta_prereq)
        print(f"✓ Archivo de prerrequisitos cargado: {len(df_prereq)} registros")
        break
    except Exception as e:
        print(f"Error al cargar prerrequisitos: {e}")
        print("Intente nuevamente.\n")

# Solicitar archivo de historial
while True:
    try:
        ruta_historial = historial#input("Ingrese la ruta del archivo 'historial_asignaturas.xlsx': ").strip()
        #df_historial = pd.read_excel(ruta_historial)
        df_historial = pd.read_csv(ruta_historial, sep=';')
        print(f"✓ Archivo de historial cargado: {len(df_historial)} registros")
        break
    except Exception as e:
        print(f"Error al cargar historial: {e}")
        print("Intente nuevamente.\n")




=== ANALIZADOR OPTIMIZADO DE PRERREQUISITOS CON CADENAS ===

✓ Archivo de prerrequisitos cargado: 3101 registros


C:\Users\00412\AppData\Local\Temp\ipykernel_4972\450024230.py:19: DtypeWarning: Columns (4,5,15,16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_historial = pd.read_csv(ruta_historial, sep=';')


✓ Archivo de historial cargado: 1490715 registros


In [ ]:
df_final = ejecutar_en_paralelo(df_historial, df_prereq, n_procs=8, debug_max_filas=5)

#df_final = evaluar_prerrequisitos_con_cadena(df_historial, df_prereq)#evaluar_prerrequisitos_mejorado(df_historial, df_prereq)


=== Paralelizando por PIDM con 8 procesos ===
Distribución de carga por proceso (filas): [186340, 186340, 186340, 186339, 186339, 186339, 186339, 186339] → total: 1490715
Creación de particiones: 3.0s


In [6]:
guardar_resultados(df_final)

✓ Guardando resultados en: Resultados v6/resultados_prerrequisitos_v6_cadenas_2025-09-26190355.csv ...
✓ Resultados guardados en: Resultados v6/resultados_prerrequisitos_v6_cadenas_2025-09-26190355.csv
Columnas en el archivo: 272
